In [1]:
import scanpy as sc
import pandas as pd
import numpy as np

/home/liuxiaodongLab/jiangjing/miniconda3/envs/agent_new/lib/python3.10/site-packages/torch/cuda/__init__.py:56: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/liuxiaodongLab/jiangjing/miniconda3/envs/agent_new/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/liuxiaodongLab/jiangjing/miniconda3/envs/agent_new/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)
/home/liuxiaodongLab/jiangjing/miniconda3/envs/agent_new/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel fro

In [2]:
#fetal = sc.read_h5ad("/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251021_fetal_adult_brain_merge_rename_scdevelopment/output/human_fetal_brain_preprocessed.h5ad",backed="r")
adult = sc.read_h5ad("/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/data/human_adult_brain_clean.h5ad",backed="r")
#pre = sc.read_h5ad("/storage2/liuxiaodongLab/fanxueying/developmental_atlas/code/20251015_pretrain_pre_post_model/embryo_pre_post_20251015.h5ad",backed="r")

In [8]:
def detailed_obs_summary(adata, columns_of_interest, save_path=None):
    """详细显示每个列的统计信息，分开打印，并可保存到文件"""
    
    # 如果提供了保存路径，创建文件并写入
    if save_path:
        with open(save_path, 'w', encoding='utf-8') as f:
            f.write("单细胞数据注释列统计汇总\n")
            f.write("=" * 50 + "\n\n")
    
    for col in columns_of_interest:
        output_lines = []
        output_lines.append(f"\n{'='*60}")
        output_lines.append(f"列名: {col}")
        output_lines.append(f"{'='*60}")
        
        if col not in adata.obs.columns:
            output_lines.append("❌ 该列不存在于数据集中")
            # 打印到屏幕
            print("\n".join(output_lines))
            # 保存到文件
            if save_path:
                with open(save_path, 'a', encoding='utf-8') as f:
                    f.write("\n".join(output_lines) + "\n")
            continue
            
        value_counts = adata.obs[col].value_counts()
        total_cells = len(adata.obs[col])
        missing_count = adata.obs[col].isnull().sum()
        
        output_lines.append(f"数据类型: {adata.obs[col].dtype}")
        output_lines.append(f"唯一值数量: {len(value_counts)}")
        output_lines.append(f"总细胞数: {total_cells}")
        output_lines.append(f"缺失值数量: {missing_count}")
        
        if missing_count > 0:
            output_lines.append(f"有效细胞数: {total_cells - missing_count}")
        
        output_lines.append(f"\n所有类别及其数量:")
        output_lines.append("-" * 40)
        for i, (category, count) in enumerate(value_counts.items(), 1):
            percentage = (count / total_cells) * 100
            output_lines.append(f"{i:2d}. {category:<30} {count:>8} cells ({percentage:6.2f}%)")
        
        # 显示前5大类别占比统计
        top5_total = value_counts.head(5).sum()
        top5_percentage = (top5_total / total_cells) * 100
        output_lines.append(f"\n前5大类别共包含: {top5_total} cells ({top5_percentage:.2f}%)")
        
        # 打印到屏幕
        print("\n".join(output_lines))
        
        # 保存到文件
        if save_path:
            with open(save_path, 'a', encoding='utf-8') as f:
                f.write("\n".join(output_lines) + "\n")



In [ ]:
# 使用函数并保存结果
detailed_obs_summary(adult, 
                    ['stage','orig_anno', 'orig_sub_anno', 'sample', 'reanno', 'orig.ident', 'lineage'],
                    save_path='/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/output/before_filter/adult_obs_summary.txt')

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import warnings
from scipy import sparse
import gc
import os
warnings.filterwarnings('ignore')

def optimize_adata_memory(adata, name=""):
    """深度优化AnnData内存使用"""
    print(f"优化 {name}: {adata.n_obs} cells, {adata.n_vars} genes")
    
    # 1. 确保X是稀疏矩阵
    if not sparse.issparse(adata.X):
        adata.X = sparse.csr_matrix(adata.X)
        print("  - X转换为稀疏矩阵")
    
    # 2. 删除不必要的部分
    if hasattr(adata, 'raw') and adata.raw is not None:
        del adata.raw
        print("  - 删除raw数据")
    
    # 3. 清理不必要的layers
    unnecessary_layers = ['spliced', 'unspliced']
    for layer in list(adata.layers.keys()):
        if layer in unnecessary_layers:
            del adata.layers[layer]
            print(f"  - 删除layer: {layer}")
    
    # 4. 优化obs和var数据类型
    for col in adata.obs.columns:
        if adata.obs[col].dtype == 'object':
            unique_ratio = adata.obs[col].nunique() / len(adata.obs)
            if unique_ratio < 0.5:  # 如果唯一值比例小于50%，转换为category
                adata.obs[col] = adata.obs[col].astype('category')
    
    for col in adata.var.columns:
        if adata.var[col].dtype == 'object':
            adata.var[col] = adata.var[col].astype('category')
    
    # 5. 强制垃圾回收
    gc.collect()
    
    return adata

def proportional_sampling_by_reanno(adata, target_cells):
    """纯比例抽样 + 内存优化"""
    
    cell_types = adata.obs['reanno'].unique()
    sampled_cells = []
    
    total_actual_cells = len(adata)
    value_counts = adata.obs['reanno'].value_counts()
    
    print(f"开始抽样，目标: {target_cells}")
    
    # 分批处理，避免内存堆积
    for i, cell_type in enumerate(cell_types):
        if i % 10 == 0:  # 每处理10个类型清理一次内存
            gc.collect()
            
        type_cells = adata[adata.obs['reanno'] == cell_type]
        original_count = len(type_cells)
        
        # 纯比例计算
        target_proportion = original_count / total_actual_cells
        type_target = int(target_cells * target_proportion)
        
        print(f"处理 {cell_type}: 原始{original_count} -> 目标{type_target}")
        
        if original_count <= type_target:
            sampled_cells.append(type_cells)
        else:
            sampled = type_cells[np.random.choice(type_cells.obs_names, type_target, replace=False)]
            sampled_cells.append(sampled)
    
    # 合并并立即优化
    adata_sampled = sc.concat(sampled_cells, join='inner')
    adata_sampled = optimize_adata_memory(adata_sampled, "抽样后")
    
    return adata_sampled

def prepare_orig_ident_column(adata, dataset_name):
    """准备orig.ident列，确保所有数据集都有统一的orig.ident列"""
    print(f"准备 {dataset_name} 的orig.ident列")
    
    # 检查现有的orig.ident列
    if 'orig.ident' in adata.obs.columns:
        print(f"  - 已存在orig.ident列")
        # 确保是字符串类型
        adata.obs['orig.ident'] = adata.obs['orig.ident'].astype(str)
    else:
        # 如果没有orig.ident列，根据数据集类型创建
        if dataset_name == 'pre':
            print(f"  - Pre数据: 使用现有的orig.ident列")
            # Pre数据应该已经有orig.ident列
            if 'orig.ident' not in adata.obs.columns:
                # 如果确实没有，创建一个基于数据集名称的
                adata.obs['orig.ident'] = dataset_name
                print(f"  - 警告: Pre数据没有orig.ident列，使用数据集名称")
        else:
            # 对于adult和fetal数据，使用dataset列作为orig.ident
            if 'orig.ident' in adata.obs.columns:
                adata.obs['orig.ident'] = adata.obs['orig.ident'].astype(str)
                print(f"  - {dataset_name}数据: orig.ident.ident")
            else:
                adata.obs['orig.ident'] = dataset_name
                print(f"  - {dataset_name}数据: 使用数据集名称作为orig.ident")
    
    print(f"  - orig.ident列数据类型: {adata.obs['orig.ident'].dtype}")
    print(f"  - orig.ident唯一值: {adata.obs['orig.ident'].nunique()}")
    
    return adata

# 设置随机种子
np.random.seed(42)

# 1. 分批加载和处理数据
print("步骤1: 分批处理数据...")

# 先只加载metadata计算共同基因
print("计算共同基因...")
fetal_genes = set(sc.read_h5ad("/storage2/liuxiaodongLab/fanxueying/developmental_atlas/code/20251107_human_fetal_brain_data_prep/human_fetal_brain_clean.h5ad", backed='r').var_names)
adult_genes = set(sc.read_h5ad("/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/data/human_adult_brain_clean.h5ad", backed='r').var_names)
pre_genes = set(sc.read_h5ad("/storage2/liuxiaodongLab/fanxueying/developmental_atlas/code/20251015_pretrain_pre_post_model/embryo_pre_post_20251015.h5ad", backed='r').var_names)

common_genes = fetal_genes & adult_genes & pre_genes
print(f"共同基因: {len(common_genes)}")

# 2. 大幅降低抽样目标
total_target = 100000  

pre_target = 35177  # Pre数据实际数量
remaining_target = total_target - pre_target
adult_target = int(remaining_target * 0.5)
fetal_target = remaining_target - adult_target

print(f"目标分配: Adult={adult_target}, Fetal={fetal_target}, Pre={pre_target}")

# 3. 逐个处理并立即保存临时文件
print("\n逐个处理数据集...")

def process_dataset(name, path, target_cells, common_genes, temp_dir="/tmp"):
    """处理单个数据集并保存临时文件"""
    print(f"\n处理 {name} 数据...")
    
    # 加载数据
    adata = sc.read_h5ad(path)
    
    # 准备orig.ident列
    adata = prepare_orig_ident_column(adata, name)
    
    # lineage处理
    if name.lower() in ['adult', 'fetal'] and 'lineage_pred' in adata.obs.columns:
        adata.obs['lineage'] = adata.obs['lineage_pred'].copy()
        adata.obs = adata.obs.drop(columns=['lineage_pred'])
    
    # 抽样
    if target_cells < len(adata):
        adata_sampled = proportional_sampling_by_reanno(adata, target_cells)
    else:
        adata_sampled = adata
    
    # 只保留共同基因
    genes_to_keep = list(set(adata_sampled.var_names) & common_genes)
    adata_sampled = adata_sampled[:, genes_to_keep].copy()
    
    # 深度优化
    adata_sampled = optimize_adata_memory(adata_sampled, f"{name}最终")
    
    # 保存临时文件
    temp_path = f"{temp_dir}/{name}_sampled_optimized.h5ad"
    adata_sampled.write_h5ad(temp_path, compression='gzip')
    
    print(f"保存到: {temp_path} (优化后)")
    
    # 清理内存
    del adata, adata_sampled
    gc.collect()
    
    return temp_path

# 处理每个数据集
temp_files = []
for name, path, target in [
    ('adult', "/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/data/human_adult_brain_clean.h5ad", adult_target),
    ('fetal', "/storage2/liuxiaodongLab/fanxueying/developmental_atlas/code/20251107_human_fetal_brain_data_prep/human_fetal_brain_clean.h5ad", fetal_target),
    ('pre', "/storage2/liuxiaodongLab/fanxueying/developmental_atlas/code/20251015_pretrain_pre_post_model/embryo_pre_post_20251015.h5ad", pre_target)
]:
    temp_file = process_dataset(name, path, target, common_genes)
    temp_files.append((name, temp_file))

# 4. 合并并最终保存
print("\n合并数据集...")
datasets = []
for name, temp_path in temp_files:
    print(f"加载 {name}...")
    adata = sc.read_h5ad(temp_path)
    adata.obs['dataset'] = name
    datasets.append(adata)

# 直接合并，不使用额外的dataset标签
combined = sc.concat(datasets, join='inner')

# 最终优化
combined = optimize_adata_memory(combined, "最终合并")

print(f"\n最终数据集: {combined.n_obs} cells, {combined.n_vars} genes")

# 保存前修复所有可能的数据类型问题
print("修复数据格式问题...")

# 修复分类数据类型问题：将所有分类列转换为字符串
for col in combined.obs.columns:
    if pd.api.types.is_categorical_dtype(combined.obs[col]):
        print(f"将分类列 '{col}' 转换为字符串")
        combined.obs[col] = combined.obs[col].astype(str)

# 检查并修复percent.mt列
if 'percent.mt' in combined.obs.columns:
    print("修复percent.mt列的数据类型")
    combined.obs['percent.mt'] = pd.to_numeric(combined.obs['percent.mt'], errors='coerce')

# 检查其他可能的数值列
numeric_columns = ['n_genes', 'n_counts', 'total_counts']  # 常见的数值列
for col in numeric_columns:
    if col in combined.obs.columns:
        combined.obs[col] = pd.to_numeric(combined.obs[col], errors='coerce')

# 确保orig.ident列存在且是字符串类型
if 'orig.ident' in combined.obs.columns:
    combined.obs['orig.ident'] = combined.obs['orig.ident'].astype(str)
    print("orig.ident列信息:")
    print(combined.obs['orig.ident'].value_counts())
else:
    print("警告: 合并后的数据没有orig.ident列")

# 然后再保存
output_path = "/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/output/filter_combined_sampled_optimized_counts.h5ad"
combined.write_h5ad(output_path)
print(f"保存优化后的数据到: {output_path}")

# 清理临时文件
for name, temp_path in temp_files:
    if os.path.exists(temp_path):
        os.remove(temp_path)
        print(f"删除临时文件: {temp_path}")

print("处理完成!")

步骤1: 分批处理数据...
计算共同基因...


In [5]:
print(combined.layers["counts"])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1968361659 stored elements and shape (499963, 27675)>
  Coords	Values
  (0, 23065)	1.0
  (0, 15545)	1.0
  (0, 6901)	1.0
  (0, 15137)	1.0
  (0, 5870)	1.0
  (0, 8029)	1.0
  (0, 11085)	2.0
  (0, 24866)	1.0
  (0, 3272)	1.0
  (0, 306)	1.0
  (0, 11247)	1.0
  (0, 9639)	1.0
  (0, 8162)	1.0
  (0, 8841)	1.0
  (0, 23297)	1.0
  (0, 6192)	1.0
  (0, 20455)	1.0
  (0, 360)	4.0
  (0, 11262)	1.0
  (0, 1442)	1.0
  (0, 5693)	1.0
  (0, 23036)	4.0
  (0, 1066)	1.0
  (0, 11549)	1.0
  (0, 1261)	1.0
  :	:
  (499962, 12004)	1165.0
  (499962, 12980)	63.0
  (499962, 12851)	26.0
  (499962, 12542)	2060.0
  (499962, 11653)	7944.0
  (499962, 4820)	2935.0
  (499962, 18191)	58.0
  (499962, 17748)	47.0
  (499962, 24060)	152137.0
  (499962, 26594)	400793.0
  (499962, 7473)	238686.0
  (499962, 23217)	1.0
  (499962, 13477)	241173.0
  (499962, 11968)	300634.0
  (499962, 6722)	1.0
  (499962, 4385)	198534.0
  (499962, 17639)	1430.0
  (499962, 8178)	112760.0
  (49996

In [42]:
combined.obs["lineage"].value_counts()

lineage
Neuron              280677
Radial glia          47870
Oligo                46312
Neuroblast           26305
TE_TrB               15935
Astrocyte            15072
Glioblast            12192
meso_Exe.meso        11466
Neuronal IPC         10152
OPC                   9781
Microglia             8488
Fibroblast            2508
epi                   2340
Vascular              1999
neural_ecto           1957
Immune                1091
hemogenic             1002
Erythrocyte            824
Choroid plexus         812
NMP                    812
ExE_endo               755
Ependymal              543
Amniotic_ecto          321
Primitive.streak       167
Endoderm               145
Notochord               91
Inner Cell Mass         88
Placodes                80
Neural crest            80
Prelineage              39
8C                      30
Morula                  19
PGC                     10
Name: count, dtype: int64

In [22]:
# 查看所有层
print(pre.layers.keys())

# 查看counts层
if 'counts' in pre.layers:
    counts_matrix = pre.layers['counts']
    print(counts_matrix[0:5, 0:5])  # 查看前5行5列
    print(f"矩阵维度: {counts_matrix.shape}")
    
    # 如果是稀疏矩阵，转换为稠密矩阵查看
    print(counts_matrix[:5, :5].toarray())
else:
    print("没有找到counts层")

# 查看X矩阵（默认矩阵）
print("X矩阵（通常是logcounts）:")
print(pre.X[0:5, 0:5].toarray() if hasattr(pre.X, 'toarray') else fetal.X[0:5, 0:5])

KeysView(Layers with keys: counts, logcounts)
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3 stored elements and shape (5, 5)>
  Coords	Values
  (0, 3)	3.0
  (1, 3)	4.0
  (2, 3)	4.0
矩阵维度: (35177, 45798)
[[0. 0. 0. 3. 0.]
 [0. 0. 0. 4. 0.]
 [0. 0. 0. 4. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]
X矩阵（通常是logcounts）:
[[0.         0.         0.         0.24291042 0.        ]
 [0.         0.         0.         0.19243727 0.        ]
 [0.         0.         0.         0.19491878 0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]]


In [7]:
combined = sc.read_h5ad("/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/output/combined_sampled_optimized_counts.h5ad")

In [41]:
# 从combined数据集中筛选出adult和fetal数据
adult_from_combined = combined[combined.obs['dataset'] == 'adult'].copy()
fetal_from_combined = combined[combined.obs['dataset'] == 'fetal'].copy()
pre_from_combined = combined[combined.obs['dataset'] == 'pre'].copy()

print("从combined数据集中提取的各数据集信息:")
print(f"Adult: {adult_from_combined.n_obs} 细胞")
print(f"Fetal: {fetal_from_combined.n_obs} 细胞")
print(f"Pre: {pre_from_combined.n_obs} 细胞")

# 使用您的detailed_obs_summary函数查看详细信息
def detailed_obs_summary(adata, columns_of_interest, save_path=None):
    """详细显示每个列的统计信息，分开打印，并可保存到文件"""
    
    # 如果提供了保存路径，创建文件并写入
    if save_path:
        with open(save_path, 'w', encoding='utf-8') as f:
            f.write("单细胞数据注释列统计汇总\n")
            f.write("=" * 50 + "\n\n")
    
    for col in columns_of_interest:
        output_lines = []
        output_lines.append(f"\n{'='*60}")
        output_lines.append(f"列名: {col}")
        output_lines.append(f"{'='*60}")
        
        if col not in adata.obs.columns:
            output_lines.append("❌ 该列不存在于数据集中")
            # 打印到屏幕
            print("\n".join(output_lines))
            # 保存到文件
            if save_path:
                with open(save_path, 'a', encoding='utf-8') as f:
                    f.write("\n".join(output_lines) + "\n")
            continue
            
        value_counts = adata.obs[col].value_counts()
        total_cells = len(adata.obs[col])
        missing_count = adata.obs[col].isnull().sum()
        
        output_lines.append(f"数据类型: {adata.obs[col].dtype}")
        output_lines.append(f"唯一值数量: {len(value_counts)}")
        output_lines.append(f"总细胞数: {total_cells}")
        output_lines.append(f"缺失值数量: {missing_count}")
        
        if missing_count > 0:
            output_lines.append(f"有效细胞数: {total_cells - missing_count}")
        
        output_lines.append(f"\n所有类别及其数量:")
        output_lines.append("-" * 40)
        for i, (category, count) in enumerate(value_counts.items(), 1):
            percentage = (count / total_cells) * 100
            output_lines.append(f"{i:2d}. {category:<30} {count:>8} cells ({percentage:6.2f}%)")
        
        # 显示前5大类别占比统计
        top5_total = value_counts.head(5).sum()
        top5_percentage = (top5_total / total_cells) * 100
        output_lines.append(f"\n前5大类别共包含: {top5_total} cells ({top5_percentage:.2f}%)")
        
        # 打印到屏幕
        print("\n".join(output_lines))
        
        # 保存到文件
        if save_path:
            with open(save_path, 'a', encoding='utf-8') as f:
                f.write("\n".join(output_lines) + "\n")

# 查看adult数据
print("="*80)
print("ADULT数据信息 (从combined中提取)")
print("="*80)
detailed_obs_summary(adult_from_combined, 
                    ['stage','orig_anno', 'orig_sub_anno', 'sample', 'reanno', 'orig.ident', 'lineage'],
                    save_path='/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/output/filter_after_summury/adult_from_combined_summary.txt')

print("\n" + "="*80)
print("FETAL数据信息 (从combined中提取)")
print("="*80)
detailed_obs_summary(fetal_from_combined,
                    ['stage','orig_anno', 'orig_sub_anno', 'sample', 'reanno', 'orig.ident', 'lineage'],
                    save_path='/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/output/filter_after_summury/fetal_from_combined_summary.txt')

# 查看细胞类型分布
print("\n" + "="*80)
print("ADULT抽样后细胞类型分布")
print("="*80)
adult_reanno_counts = adult_from_combined.obs['reanno'].value_counts()
for cell_type, count in adult_reanno_counts.items():
    proportion = (count / len(adult_from_combined)) * 100
    print(f"{cell_type}: {count} cells ({proportion:.2f}%)")

print("\n" + "="*80) 
print("FETAL抽样后细胞类型分布")
print("="*80)
fetal_reanno_counts = fetal_from_combined.obs['reanno'].value_counts()
for cell_type, count in fetal_reanno_counts.items():
    proportion = (count / len(fetal_from_combined)) * 100
    print(f"{cell_type}: {count} cells ({proportion:.2f}%)")

# 简洁统计
print("\n" + "="*80)
print("各数据集最终统计")
print("="*80)
print(f"Adult: {adult_from_combined.n_obs} 细胞, {adult_from_combined.n_vars} 基因")
print(f"Fetal: {fetal_from_combined.n_obs} 细胞, {fetal_from_combined.n_vars} 基因") 
print(f"Pre: {pre_from_combined.n_obs} 细胞, {pre_from_combined.n_vars} 基因")
print(f"总计: {combined.n_obs} 细胞, {combined.n_vars} 基因")

从combined数据集中提取的各数据集信息:
Adult: 311401 细胞
Fetal: 153385 细胞
Pre: 35177 细胞
ADULT数据信息 (从combined中提取)

列名: stage
数据类型: category
唯一值数量: 4
总细胞数: 311401
缺失值数量: 0

所有类别及其数量:
----------------------------------------
 1. 50-year-old stage                119373 cells ( 38.33%)
 2. 42-year-old stage                 96340 cells ( 30.94%)
 3. 29-year-old stage                 91008 cells ( 29.23%)
 4. 60-year-old stage                  4680 cells (  1.50%)

前5大类别共包含: 311401 cells (100.00%)

列名: orig_anno
数据类型: category
唯一值数量: 10
总细胞数: 311401
缺失值数量: 0

所有类别及其数量:
----------------------------------------
 1. Cerebral cortex                  120576 cells ( 38.72%)
 2. Cerebral nuclei                   48203 cells ( 15.48%)
 3. Hippocampus                       32224 cells ( 10.35%)
 4. Thalamus                          31680 cells ( 10.17%)
 5. Midbrain                          23007 cells (  7.39%)
 6. Pons                              18930 cells (  6.08%)
 7. Cerebellum                        14173 ce

In [43]:
detailed_obs_summary(pre_from_combined,
                    ['stage','orig_anno', 'orig_sub_anno', 'sample', 'reanno', 'orig.ident', 'lineage'],
                    save_path='/storage2/liuxiaodongLab/jiangjing/Projects/XueyingFan/PD_XueyingFan/20251106_fetal_adult_merge_name_scdevelopment/output/filter_after_summury/pre_from_combined_summary.txt')


列名: stage
数据类型: category
唯一值数量: 23
总细胞数: 35177
缺失值数量: 0

所有类别及其数量:
----------------------------------------
 1. CS10                              13774 cells ( 39.16%)
 2. E14_IVC                            3339 cells (  9.49%)
 3. E9_IVC                             3270 cells (  9.30%)
 4. E13_IVC                            3148 cells (  8.95%)
 5. E11_IVC                            2859 cells (  8.13%)
 6. E8_IVC                             2194 cells (  6.24%)
 7. E10_IVC                            1669 cells (  4.74%)
 8. E12_IVC                            1393 cells (  3.96%)
 9. CS7                                1189 cells (  3.38%)
10. E6_IVC                              730 cells (  2.08%)
11. 7.0                                 493 cells (  1.40%)
12. 6.0                                 469 cells (  1.33%)
13. 5.0                                 333 cells (  0.95%)
14. CS9                                 162 cells (  0.46%)
15. E7_IVC                               58 cells (